In [1]:
import os
import sys
import torch

# Add src directory to path
sys.path.insert(0, os.path.abspath('../src'))

from parser import process_pdf
from rag import *
from section_splitter import split_into_sections
from section_classifier import embed, TARGET_QUERIES
from structure_chunker import *
from structured_retriever import *
from agents import *
from multiagent import *
from evaluation import (
    llm_as_judge_evaluation, 
    print_evaluation_report, 
    evaluate_all_agents,
    print_summary_report,
    extract_agent_prompts
)

/home/ixti95/Clinical-Trial-Protocol-Design-Support/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/ixti95/Clinical-Trial-Protocol-Design-Support/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/home/

In [2]:
parsed_data = process_pdf('../data/protocol.pdf')

In [3]:
chunks=build_structured_chunks(parsed_data)

In [4]:
# Agent configuration mapping
FUNCTIONS = {
    "objectives and endpoints":extract_objectives,
    "eligibility": extract_eligibility,
    "schedule of activities":extract_soa,
    "visit definitions": extract_visit_definitions,
    "key assessments": extract_key_assessments
}


def run_agent_extraction(chunks, agent_name, num_sections=3):
    """
    Run extraction for a specific agent.
    
    Args:
        chunks: List of chunk objects with attributes like section_id, full_title, content
        agent_name: One of "objectives and endpoints", "eligibility", "schedule of activities", "visit definitions", "key assessments"
        num_sections: Number of top sections to retrieve
    
    Returns:
        Validated Pydantic model output from the agent
    """
    if agent_name not in FUNCTIONS:
        raise ValueError(f"Unknown agent: {agent_name}. Must be one of {list(FUNCTIONS.keys())}")

    extraction_func = FUNCTIONS[agent_name]
    
    # Get top relevant sections
    retriever = BM25StructuredRetriever(chunks, TARGET_QUERIES)
    top_sections = retriever.retrieve_context(agent_name, top_k=num_sections)
    
    # Run extraction
    return extraction_func(top_sections)

def find_top_sections(chunks, agent_name, num_sections=3):

    if agent_name not in FUNCTIONS:
        raise ValueError(f"Unknown agent: {agent_name}. Must be one of {list(FUNCTIONS.keys())}")

    # Get top relevant sections
    retriever = BM25StructuredRetriever(chunks, TARGET_QUERIES)
    top_sections = retriever.retrieve_chunks(agent_name, top_k=num_sections)
    
    return top_sections

# Testing agents, rag, and multiagent

In [ ]:
objectives_output = run_agent_extraction(chunks, "objectives and endpoints", num_sections=2)
# eligibility_output = run_agent_extraction(chunks, "eligibility", num_sections=2)
# soa_output = run_agent_extraction(chunks, "schedule of activities", num_sections=2)
# visit_definitions_output = run_agent_extraction(chunks, "visit definitions", num_sections=2)
# key_assessments_output = run_agent_extraction(chunks, "key assessments", num_sections=5)

In [ ]:
# json printing for better readability
print(json.dumps(json.loads(objectives_output.model_dump_json()), indent=4))

In [5]:
rag = ClinicalProtocolRAG(parsed_data)

question = "What is the placebo in the study?"

answer = rag.answer(question)
print("Answer:", answer)

Token indices sequence length is longer than the specified maximum sequence length for this model (2615 > 512). Running this sequence through the model will result in indexing errors


Building structured chunks...
Building QA documents...
Building FAISS index...


/home/ixti95/Clinical-Trial-Protocol-Design-Support/src/rag.py:121: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Answer: Saline placebo.



In [8]:
# Convert chunks to sections dictionary for supervisor
sections_dict = {}
for chunk in chunks:
    # Use full_title as key and content as value
    if hasattr(chunk, 'full_title') and hasattr(chunk, 'content'):
        sections_dict[chunk.full_title] = chunk.content
    elif hasattr(chunk, 'section_id') and hasattr(chunk, 'content'):
        sections_dict[chunk.section_id] = chunk.content

print(f"Created {len(sections_dict)} sections from {len(chunks)} chunks")

supervisor = SupervisorMultiAgent(
    sections=sections_dict,
    parsed_text=parsed_data)

# Ask a question
# query = "Perform data validation with the eligibility criteria?"
# response_json = supervisor.answer(query)
# response = json.loads(response_json)

# print(f"Route: {response['route']}")
# print(f"Routing Info: {response.get('routing_info', {})}")


Created 61 sections from 61 chunks
Building structured chunks...
Building QA documents...
Building FAISS index...


# Evaluation

## Supervisor Evaluation

In [9]:
# Create labeled dataset with query-route pairs
labeled_data = [
    {"query": "What are the primary objectives?", "route": "objectives and endpoints"},
    {"query": "What are the secondary endpoints?", "route": "objectives and endpoints"},
    {"query": "List inclusion criteria", "route": "eligibility"},
    {"query": "What are the exclusion criteria?", "route": "eligibility"},
    {"query": "Show me the schedule of activities", "route": "schedule of activities"},
    {"query": "What procedures are done at Day 1?", "route": "schedule of activities"},
    {"query": "How is screening visit defined?", "route": "visit definitions"},
    {"query": "When is the final visit?", "route": "visit definitions"},
    {"query": "What are the safety assessments?", "route": "key assessments"},
    {"query": "List all assessments and their procedures", "route": "key assessments"},
    {"query": "Perform data validation with the eligibility criteria?", "route": "eligibility check"},
    {"query": "Check if patients meet eligibility requirements", "route": "eligibility check"},
    {"query": "Give me a summary of the protocol", "route": "rag"},
    {"query": "What is the study duration?", "route": "rag"},
]

print(f"Created labeled dataset with {len(labeled_data)} examples")

Created labeled dataset with 14 examples


In [11]:
# Evaluate supervisor routing accuracy
correct = 0
total = len(labeled_data)
results = []

for item in labeled_data:
    query = item["query"]
    expected_route = item["route"]
    
    # Get supervisor routing decision
    response_json = supervisor.answer(query)
    response = json.loads(response_json)
    predicted_route = response['route']
    
    match = predicted_route == expected_route
    if match:
        correct += 1
    
    results.append({
        "query": query,
        "expected": expected_route,
        "predicted": predicted_route,
        "match": match
    })

# Calculate accuracy
accuracy = correct / total if total > 0 else 0.0

print(f"\n{'='*60}")
print(f"SUPERVISOR ROUTING EVALUATION")
print(f"{'='*60}")
print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
print(f"{'='*60}\n")

# Show results
for i, result in enumerate(results, 1):
    status = "✓" if result["match"] else "✗"
    print(f"{status} [{i:2d}] {result['query'][:40]:40} | Expected: {result['expected']:15} | Got: {result['predicted']:15}")


SUPERVISOR ROUTING EVALUATION
Accuracy: 100.00% (14/14)

✓ [ 1] What are the primary objectives?         | Expected: objectives and endpoints | Got: objectives and endpoints
✓ [ 2] What are the secondary endpoints?        | Expected: objectives and endpoints | Got: objectives and endpoints
✓ [ 3] List inclusion criteria                  | Expected: eligibility     | Got: eligibility    
✓ [ 4] What are the exclusion criteria?         | Expected: eligibility     | Got: eligibility    
✓ [ 5] Show me the schedule of activities       | Expected: schedule of activities | Got: schedule of activities
✓ [ 6] What procedures are done at Day 1?       | Expected: schedule of activities | Got: schedule of activities
✓ [ 7] How is screening visit defined?          | Expected: visit definitions | Got: visit definitions
✓ [ 8] When is the final visit?                 | Expected: visit definitions | Got: visit definitions
✓ [ 9] What are the safety assessments?         | Expected: key assessments | 

## Agent Section Retrieval Evaluation (Recall)

In [12]:
# Define ground truth sections for each agent
# These are the section IDs or titles that should be retrieved for each agent task
ground_truth_sections = {
    "objectives and endpoints": ["OBJECTIVES   AND ENDPOINTS"],
    "eligibility": ["STUDY  POPULATION > Inclusion Criteria", "STUDY  POPULATION > Exclusion Criteria"],
    "schedule of activities": ["PROTOCOL   SUMMARY > Schedule of Activities"],
    "key assessments": ["STUDY  ASSESSMENTS   AND  PROCEDURES > Efficacy Assessments", "STUDY  ASSESSMENTS   AND  PROCEDURES > Adverse Events and Serious Adverse Events", "STUDY  ASSESSMENTS   AND  PROCEDURES > Safety Assessments"],
    "visit definitions": ["PROTOCOL   SUMMARY > Schedule of Activities"],
}

# Retrieve sections for each agent and compute recall

retrieval_results = {}

for agent_name, ground_truth in ground_truth_sections.items():

    retrieved_chunks = find_top_sections(chunks, agent_name, num_sections=3)
    
    # Extract section IDs/titles from retrieved chunks
    retrieved_sections = set()
    for chunk in retrieved_chunks:
        # Normalize section titles for comparison
        normalized = chunk.full_title.lower().strip()
        retrieved_sections.add(normalized)
    
    # Normalize ground truth for comparison
    normalized_ground_truth = set(s.lower().strip() for s in ground_truth)
    
    # Calculate recall
    if len(normalized_ground_truth) > 0:
        recall = len(retrieved_sections & normalized_ground_truth) / len(normalized_ground_truth)
    else:
        recall = 0.0
    
    # Store results
    retrieval_results[agent_name] = {
        "recall": recall,
        "retrieved": len(retrieved_sections),
        "ground_truth": len(ground_truth),
        "matching": len(retrieved_sections & normalized_ground_truth),
    }
    
   
# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
for agent_name, results in retrieval_results.items():
    print(f"{agent_name:20} | Recall: {results['recall']:.2%} ({results['matching']}/{results['ground_truth']})")



SUMMARY
objectives and endpoints | Recall: 100.00% (1/1)
eligibility          | Recall: 100.00% (2/2)
schedule of activities | Recall: 100.00% (1/1)
key assessments      | Recall: 0.00% (0/3)
visit definitions    | Recall: 100.00% (1/1)


## Agent output evaluation

In [18]:
objectives_output = run_agent_extraction(chunks, "objectives and endpoints", num_sections=2)
eligibility_output = run_agent_extraction(chunks, "eligibility", num_sections=2)
soa_output = run_agent_extraction(chunks, "schedule of activities", num_sections=2)
visit_definitions_output = run_agent_extraction(chunks, "visit definitions", num_sections=2)
key_assessments_output = run_agent_extraction(chunks, "key assessments", num_sections=5)

### LLM-as-Judge Evaluation

We'll evaluate each agent's output using an LLM-as-judge approach. The LLM will be given:
- The full protocol document
- The original prompt used by the agent
- The agent's output

The LLM will evaluate:
- **Completeness**: Did the agent extract all relevant information?
- **Accuracy**: Is the extracted information correct?
- **Format Compliance**: Does the output follow the expected structure?

In [19]:
# Collect agent outputs
agent_outputs = {
    "objectives and endpoints": objectives_output,
    "eligibility": eligibility_output,
    "schedule of activities": soa_output,
    "visit definitions": visit_definitions_output,
    "key assessments": key_assessments_output
}

# Get the agent prompts
agent_prompts = extract_agent_prompts()

In [20]:
# Run LLM-as-judge evaluation for all agents
evaluations = evaluate_all_agents(
    full_document=parsed_data,
    agent_outputs=agent_outputs,
    prompts=agent_prompts
)

print("\nEvaluation complete!")

Evaluating objectives and endpoints...
Evaluating eligibility...
Evaluating schedule of activities...
Evaluating visit definitions...
Evaluating key assessments...

Evaluation complete!


In [21]:
# Display summary report
print_summary_report(evaluations)


EVALUATION SUMMARY
Agent                             Overall   Complete   Accurate     Format
--------------------------------------------------------------------------------
objectives and endpoints             7.7       7.0       8.0       8.0
eligibility                          8.7       9.0       9.0       8.0
schedule of activities               6.4       6.0       7.0       6.3
visit definitions                    6.8       7.0       7.0       6.5
key assessments                      6.7       5.0       8.0       7.0


In [22]:
# Display detailed report for each agent
for agent_name, evaluation in evaluations.items():
    print_evaluation_report(evaluation)
    print()  # Add spacing between reports

EVALUATION REPORT: OBJECTIVES AND ENDPOINTS

SCORES:
  Overall:            7.7/10
  Completeness:       7.0/10
  Accuracy:           8.0/10
  Format Compliance:  8.0/10

REASONING:
  The agent did a good job overall of extracting the objectives and endpoints from the protocol. It missed some key instructions in the prompt though, namely that it should not invent objectives, that each objective must have at least one endpoint, and it invented objectives by splitting apart objectives that were actually endpoints related to the primary objective. It also missed one exploratory objective at the end of the table.

STRENGTHS:
  ✓ Accurately extracted most objectives and endpoints.
  ✓ Maintained the wording from the original document.
  ✓ Successfully categorized most objectives correctly.
  ✓ Output is generally well formatted and parsable as JSON.

WEAKNESSES:
  ✗ Failed to follow the prompt's instructions: "Do NOT invent objectives or endpoints". Several objectives are endpoint items.
  ✗